In [ ]:
# Для установки библиотеки воспользуйтесь командой
# !conda install conda-forge::transformers
# или
# !pip install transformers

In [ ]:
# Любая LLM работает с токенами, поэтому нам нужна не только модель, но и токенизатор
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Зададим название модели из репозитория huggingface
# список доступных моделей можно посмотреть по ссылке https://huggingface.co/models
# будем использовать модель Qwen или YandexGPT

model_name = "IlyaGusev/saiga_llama3_8b"
#model_name = "yandex/YandexGPT-5-Lite-8B-instruct"

# В названии модели обычно указываются - версия (2,5) количество параметров (7 млрд.)
# этап обучения или предназначение (Base, Instruct, Code, Thinking и т.п.)
# размер контекстного окна (1 млн.)
# также могут указываться параметры квантования (FP8, GGUF, ) и другие характеристики.


# Загружаем предобученную можель
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# Загружаем предобученный токенизатор
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/277 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

In [ ]:
# Загружаем текстовый файл преобразуя его в строку
file = 'ENG_article.txt'
with open(file, 'r', encoding='cp1251') as file:
    data = file.read().replace('\n', ' ')

print(len(data))

8291


In [ ]:
# Посмотрим выдержку из файла
print(data[90:160])

 (machine learning)     This article is about the computational models


In [ ]:
# Посчитаем количество слов в файле чтобы примерно сопоставить с контекстным окном
num_words = len(data.split())
print(num_words)

1157


In [ ]:
# Для общения с LLM нужно составить промпт
# он будет состоять из системной части - наши инструкции что нужно сделать
# и пользовательской части - текста файла
questions = """
1. В каком году была обозначена проблема взрывающихся градиентов?
2. Кто в 1891 году разработал метод уничтожающей производной?
3. Кто предложил цепное правило дифференцирования и в каком году?
"""
messages = [
    {
        "role": "system",
        "content": (
            "Ты строгий аналитик. Твоя задача — отвечать на вопросы ТОЛЬКО на основе предоставленного текста. "
            "Категорически запрещено использовать твои внутренние знания. "
            "Если в тексте нет прямого ответа на вопрос, ты обязан ответить: 'В тексте нет информации об этом'. "
            "Отвечай на русском языке."
        )
    },
    {
        "role": "user",
        "content": f"Текст:\n{data}\n\nВопросы:\n{questions}"
    }
]

# Для YandexGPT инструкции пишутся от имени пользователя
#messages = [
#    {"role": "user", "content": "Суммаризируй текст. Определи жанр текста, выдели основную информацию. Ответ давай на русском языке."+data[0:5000]} # берем только часть чтобы долго не ждать генерацию
#]

# Метод apply_chat_template() используется для форматирования сообщений в одну строку, формат
# которой ориентирован на использование чат-ориентированных языковых моделей.
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# В текст добавляются специальные  токены-метки для указания структуры разговора.
print(text[0:160])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Ты строгий аналитик. Твоя задача — отвечать на вопросы ТОЛЬКО на основе предоставленного текста. Ка


In [ ]:
# Токенизатор разбивает текст на токены.
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# input_ids — токены в числовом виде, которые представляют собой уникальный номер (ID) из словаря модели.
# attention_mask — задает маску, которая показывает позиции реальных токенов и дополнений (padding) или служебных токенов.
# Модели обрабатывают батчи текстов одинаковой длины. Чтобы выровнять длину реальных текстов короткие тексты дополняются специальным токеном [PAD].
# Таким токенам в attention_mask соответствуют нули, чтобы модель их не обрабатывала.
print(model_inputs)

{'input_ids': tensor([[128000, 128000, 128006,  ...,  78191, 128007,    271]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0')}


In [ ]:
# Токенизированный текст подаем в модель.
# max_new_tokens - задает максимальное число генерируемых в ответе токенов.
# Можно указать и другие параметры:
# temperature - регулирует "случайность" в выборе следующего токена (<1 — более предсказуемо, >1 — более хаотично).
# top_k - ограничивает выбор следующего токена K наиболее вероятными
# repetition_penalty - штраф за повторяющиеся токены (>1 - штраф, 1 - без штрафа).
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=1024,
#    temperature=0.9,
#    top_k=50,
#    repetition_penalty=1.2
)

In [ ]:
# Так как модель возвращает и промпт и сгенерированные токены, выделяем только ответ.
generated_ids_ = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# Преобразуем ID токенов обратно в слова.
response = tokenizer.batch_decode(generated_ids_, skip_special_tokens=True)[0]

In [ ]:
# Смотрим что получилось.
print(response)

1. Проблема взрывающихся градиентов была обозначена в 1991 году.
2. Метод уничтожающей производной не был разработан в 1891 году. В тексте нет упоминаний о таком методе или его авторе.
3. Цепное правило дифференцирования было предложено Готфридом Вильгельмом Лейбницем в 1673 году.
